# 01_02_remove_duplicated_fits_file

## 필요한 모듈

이 프로젝트를 위해서는 아래의 모듈이 필요하다. 

> numpy, pandas, matplotlib, astropy, version_information

### 모듈 설치

1. 콘솔 창에서 모듈을 설치할 때는 아래와 같은 형식으로 입력하면 된다.

>pip install module_name==version

>conda install module_name=version

2. 주피터 노트북(코랩 포함)에 설치 할 때는 아래의 셀을 실행해서 실행되지 않은 모듈을 설치할 수 있다. (pip 기준) 만약 아나콘다 환경을 사용한다면 7행을 콘다 설치 명령어에 맞게 수정하면 된다.

In [1]:
#import sys
#!pip install astropy==5.2 photutils==1.6 #astroscrappy==1.1.1
#%pip install astroscrappy==1.1.1

In [2]:
import importlib, sys, subprocess
packages = "numpy, pandas, matplotlib, scipy, astropy, photutils, ccdproc, astroscrappy, version_information" # required modules
pkgs = packages.split(", ")
for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        #print(f"**** module {pkg} is not installed")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    else: 
        print(f"**** module {pkg} is installed")

%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")

**** module numpy is installed
**** module pandas is installed
**** module matplotlib is installed
**** module scipy is installed
**** module astropy is installed
**** module photutils is installed
**** module ccdproc is installed
**** module astroscrappy is installed
**** module version_information is installed
This notebook was generated at 2023-11-20 08:23:19 (KST = GMT+0900) 
0 Python     3.11.5 64bit [GCC 12.3.0]
1 IPython    8.15.0
2 OS         Linux 5.15.0 88 generic x86_64 with glibc2.31
3 numpy      1.25.2
4 pandas     2.1.0
5 matplotlib 3.8.0
6 scipy      1.11.2
7 astropy    5.2.1
8 photutils  1.6.0
9 ccdproc    2.4.1
10 astroscrappy 1.1.0
11 version_information 1.0.4


### import modules

In [3]:
from glob import glob
from pathlib import Path, PosixPath, WindowsPath
import os
from astropy.time import Time
from datetime import datetime, timedelta
from astropy.io import fits
import shutil 

import ysfitsutilpy as yfu
#import ysphotutilpy as ypu
#import ysvisutilpy as yvu

import _astro_utilities
import _Python_utilities

In [4]:
#%%
#######################################################
# read all files in base directory for processing
BASEDIR = Path("/mnt/Rdata/OBS_data") 
DOINGDIR = Path(BASEDIR/ "asteroid" / "RiLA600_STX-16803_-_1bin")
#DOINGDIR = Path(BASEDIR/ "asteroid" / "GSON300_STF-8300M_-_1bin")

DOINGDIRs = sorted(_Python_utilities.getFullnameListOfsubDirs(DOINGDIR))
print ("len(DOINGDIRs): ", format(len(DOINGDIRs)))
print ("DOINGDIRs: ", format(DOINGDIRs))

len(DOINGDIRs):  77
DOINGDIRs:  ['/mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803_-_1bin/120LACHESIS_LIGHT_-_2023-10-10_-_RiLA600_STX-16803_-_1bin/', '/mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803_-_1bin/120LACHESIS_LIGHT_-_2023-10-11_-_RiLA600_STX-16803_-_1bin/', '/mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803_-_1bin/120LACHESIS_LIGHT_-_2023-10-16_-_RiLA600_STX-16803_-_1bin/', '/mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803_-_1bin/120LACHESIS_LIGHT_-_2023-10-17_-_RiLA600_STX-16803_-_1bin/', '/mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803_-_1bin/120LACHESIS_LIGHT_-_2023-11-14_-_RiLA600_STX-16803_-_1bin/', '/mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803_-_1bin/135HERTHA_LIGHT_-_2023-10-10_-_RiLA600_STX-16803_-_1bin/', '/mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803_-_1bin/135HERTHA_LIGHT_-_2023-10-11_-_RiLA600_STX-16803_-_1bin/', '/mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803_-_1bin/135HERTHA_LIGHT_-_2023-10-16_-_RiLA600_STX-16803_-_1bin/', '/mnt/Rdata/OBS_data/asteroid/RiLA600_STX-168

In [5]:
DOINGDIRs[7]

'/mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803_-_1bin/135HERTHA_LIGHT_-_2023-10-16_-_RiLA600_STX-16803_-_1bin/'

In [6]:
for DOINGDIR in DOINGDIRs[:] : 
    DOINGDIR = Path(DOINGDIR)
    print(f"Starting: {str(DOINGDIR.parts[-1])}")
    fits_in_dir = sorted(list(DOINGDIR.glob('*.fit*')))
    #print("fits_in_dir", fits_in_dir)
    print("len(fits_in_dir)", len(fits_in_dir))

    if len(fits_in_dir) == 0 :
        print(f"There is no fits fils in {DOINGDIR}")
        pass
    else : 
        summary = None 
        summary = yfu.make_summary(DOINGDIR/"*.fit*",
                    #output = save_fpath,
                    verbose = False
                    )
        print("summary: ", summary)
        print("len(summary)", len(summary))

Starting: 120LACHESIS_LIGHT_-_2023-10-10_-_RiLA600_STX-16803_-_1bin
len(fits_in_dir) 495
summary:                                                    file  filesize  SIMPLE  \
0    /mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803...  33753600    True   
1    /mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803...  33563520    True   
2    /mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803...  33753600    True   
3    /mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803...  33563520    True   
4    /mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803...  33753600    True   
..                                                 ...       ...     ...   
490  /mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803...  33560640    True   
491  /mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803...  33560640    True   
492  /mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803...  33560640    True   
493  /mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803...  33560640    True   
494  /mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803...  33560640 

In [ ]:
for _, row in summary.iterrows():
    fpath = Path(row["file"])
    print (f"starting {fpath.name}...")
    if fpath.suffix == ".fit" and \
        (fpath.parents[0]/f"{fpath.stem}.fits").exists() : 
        os.remove(fpath)
        print(f"{fpath.name} is removed")

starting 120LACHESIS_LIGHT_R_2023-10-16-15-46-33_150sec_RiLA600_STX-16803_-29c_1bin.fit...
120LACHESIS_LIGHT_R_2023-10-16-15-46-33_150sec_RiLA600_STX-16803_-29c_1bin.fit is removed
starting 120LACHESIS_LIGHT_R_2023-10-16-15-46-33_150sec_RiLA600_STX-16803_-29c_1bin.fits...
starting 120LACHESIS_LIGHT_R_2023-10-16-15-49-15_210sec_RiLA600_STX-16803_-29c_1bin.fit...
120LACHESIS_LIGHT_R_2023-10-16-15-49-15_210sec_RiLA600_STX-16803_-29c_1bin.fit is removed
starting 120LACHESIS_LIGHT_R_2023-10-16-15-49-15_210sec_RiLA600_STX-16803_-29c_1bin.fits...
starting 120LACHESIS_LIGHT_R_2023-10-16-15-52-56_150sec_RiLA600_STX-16803_-29c_1bin.fit...
120LACHESIS_LIGHT_R_2023-10-16-15-52-56_150sec_RiLA600_STX-16803_-29c_1bin.fit is removed
starting 120LACHESIS_LIGHT_R_2023-10-16-15-52-56_150sec_RiLA600_STX-16803_-29c_1bin.fits...
starting 120LACHESIS_LIGHT_R_2023-10-16-15-55-38_210sec_RiLA600_STX-16803_-29c_1bin.fit...
120LACHESIS_LIGHT_R_2023-10-16-15-55-38_210sec_RiLA600_STX-16803_-29c_1bin.fit is removed


In [5]:
for DOINGDIR in DOINGDIRs[:] : 
    DOINGDIR = Path(DOINGDIR)
    print(f"Starting: {str(DOINGDIR.parts[-1])}")
    fits_in_dir = sorted(list(DOINGDIR.glob('*.fit*')))
    #print("fits_in_dir", fits_in_dir)
    print("len(fits_in_dir)", len(fits_in_dir))

    if len(fits_in_dir) == 0 :
        print(f"There is no fits fils in {DOINGDIR}")
        pass
    else : 
        summary = None 
        summary = yfu.make_summary(DOINGDIR/"*.fit*",
                    #output = save_fpath,
                    verbose = False
                    )
        print("summary: ", summary)
        print("len(summary)", len(summary))

        for _, row in summary.iterrows():
            fpath = Path(row["file"])
            print (f"starting {fpath.name}...")
            if {fpath.suffix} == ".fit" and \
                (fpath.parents[0]/f"{fpath.stem}.fits").exists() : 
                os.remove(fpath)
                print(f"{fpath.name} is removed")

Starting: 120LACHESIS_LIGHT_-_2023-10-10_-_RiLA600_STX-16803_-_1bin
len(fits_in_dir) 495
summary:                                                    file  filesize  SIMPLE  \
0    /mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803...  33753600    True   
1    /mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803...  33563520    True   
2    /mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803...  33753600    True   
3    /mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803...  33563520    True   
4    /mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803...  33753600    True   
..                                                 ...       ...     ...   
490  /mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803...  33560640    True   
491  /mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803...  33560640    True   
492  /mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803...  33560640    True   
493  /mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803...  33560640    True   
494  /mnt/Rdata/OBS_data/asteroid/RiLA600_STX-16803...  33560640 

OSError: Empty or corrupt FITS file

In [ ]:
fpath.suffix
fpath.parent

PosixPath('/mnt/homes/parksparks/Drive/KBox/Github/astro_Python')